**<h1 align="center">ESCO Augmented Evaluation</h1>**

This notebook applies ESCO ontology–based skill features to SBERT top-K candidate lists to construct reranked datasets for career transition prediction. A fixed hybrid scoring function is first applied, followed by parameter tuning on the validation set.

The resulting reranked candidate sets are saved for downstream evaluation and analysis.

# Table of Contents
* [1. Imports & Setup](#chapter1)
* [2. Load Data](#chapter2)
* [3. Build skill label → URI dictionary](#chapter3)
* [4. Parse CV skills and build CV→skillset lookup](#chapter4)
* [5. SBERT Top-K retrieval (candidate generation)](#chapter5)
* [6. Add ESCO overlap features to candidates](#chapter6)
* [7. Evaluation metrics](#chapter7)
* [8. Hybrid-score + tuning (α grid + feature variants)](#chapter8)
* [9. Run & Save](#chapter9)
    * [9.1. Off the shelf SBERT](#sub-section-9_1)
    * [9.2. Tuned SBERT](#sub-section-9_2)
    * [9.3. Hybrid tuning SBERT off shelf](#sub-section-9_3)
    * [9.4. Save val_topk_feat_off and test_topk_feat_off](#sub-section-9_4)

<a class="anchor" id="chapter1"></a>

# 1. Imports & Setup

</a>

In [ ]:
import os
import sys
import json
import re
import math
import ast
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer, util

In [ ]:
# Project root (assumes notebook is in /Notebook)
PROJECT_ROOT = Path("..").resolve()

# Allow imports from project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from thesis_utility import save_json

In [ ]:
DATA_DIR = PROJECT_ROOT / "Data"
OUT_DIR = PROJECT_ROOT / "Results" / "Hybrid"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME = datetime.now().strftime("%m%d%Y_%H%M")

In [ ]:
MODEL_OFF_SHELF = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_TUNED = PROJECT_ROOT / "Models" / "sbert_baseline_epoch1_bs128_20260113_1453"

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [ ]:
# Load datasets produced in notebook 01
val = pd.read_csv(DATA_DIR / "Dataset" / "val.csv")
test = pd.read_csv(DATA_DIR / "Dataset" / "test.csv")

In [ ]:
# Load ESCO resources and ontology artifacts
skills = pd.read_csv(DATA_DIR / "ESCO Files" / "skills_en.csv")

In [ ]:
with open(DATA_DIR / "Ontology Artifacts" / "occ_to_skills.json", "r") as f:
    occ_to_skills = json.load(f)

with open(DATA_DIR / "Ontology Artifacts" / "skill_df.json", "r") as f:
    skill_df = json.load(f)

with open(DATA_DIR / "Ontology Artifacts" / "skill_idf.json", "r") as f:
    skill_idf = json.load(f)

with open(DATA_DIR / "Ontology Artifacts" / "occ_avg_skill_idf.json", "r") as f:
    occ_avg_skill_idf = json.load(f)

with open(DATA_DIR / "Ontology Artifacts" / "skill_to_parents.json", "r") as f:
    skill_to_parents = json.load(f)

In [ ]:
# Convert occupation–skill mapping to use sets for overlap computations
occ_to_skills = {occ: set(skills) for occ, skills in occ_to_skills.items()}

<a class="anchor" id="chapter3"></a>

# 3. Build skill label → URI dictionary

</a>

In [ ]:
# Build mapping from skill labels to skill URI
label_to_uri = {}

for _, row in skills.iterrows():
    uri = row["conceptUri"]

    # Map preferred label to URI
    pref = row.get("preferredLabel")
    if isinstance(pref, str) and pref.strip():
        label_to_uri.setdefault(pref.strip().lower(), uri)

    # Map alternative labels (split on ';' or newline) to the same URI
    altlabels = row.get("altLabels")
    if isinstance(altlabels, str):
        for alt in re.split(r"[;\n]", altlabels):
            alt = alt.strip()
            if alt:
                label_to_uri.setdefault(alt.lower(), uri)

In [ ]:
print("label_to_uri size:", len(label_to_uri))

label_to_uri size: 99618


<a class="anchor" id="chapter4"></a>

# 4. Parse CV skills and build CV→skillset lookup

</a>

In [ ]:
# Build mapping from (person_id, t) to set of ESCO skill URIs for each CV
def build_cv_skill_lookup(df_split, label_to_uri):

    lookup = {}

    for _, row in df_split.iterrows():
        key = (row["person_id"], row["t"])
        x = row["cv_skills_lol"]

        # Parse CV skills (may be list or stringified list-of-lists)
        if isinstance(x, list):
            cv_lol = x
        elif not isinstance(x, str) or x.strip() == "":
            cv_lol = []
        else:
            try:
                obj = ast.literal_eval(x)
                cv_lol = obj if isinstance(obj, list) else []
            except Exception:
                cv_lol = []

        # Map skill labels to ESCO URIs and collect unique skills
        S = set()
        for skills_list in cv_lol:
            if not isinstance(skills_list, (list, tuple)):
                continue
            for skill_label in skills_list:
                if not isinstance(skill_label, str):
                    continue
                uri = label_to_uri.get(skill_label.strip().lower())
                if uri is not None:
                    S.add(uri)

        lookup[key] = S

    return lookup

In [ ]:
x = val.loc[0, "cv_skills_lol"]
print(type(x), str(x)[:120])

# Mapping rate check
cv_lol = ast.literal_eval(x) if isinstance(x, str) else x
flat_labels = [s for sub in cv_lol for s in (sub if isinstance(sub, list) else []) if isinstance(s, str)]
mapped = sum(1 for s in flat_labels if s.strip().lower() in label_to_uri)
print("Label mapping rate:", mapped, "/", len(flat_labels), f"({mapped/max(1,len(flat_labels)):.1%})")

<class 'str'> [['clinical reports', 'administrative tasks in a medical environment', 'medical informatics', 'use spreadsheets software
Label mapping rate: 14 / 14 (100.0%)


<a class="anchor" id="chapter5"></a>

# 5. SBERT Top-K retrieval (candidate generation)

</a>

In [ ]:
# Build top-K SBERT candidate lists for each CV query in a split
def build_topk_df(df_split: pd.DataFrame, model, k: int = 50, batch_size: int = 256):
    
    # Build deduplicated job description corpus (one row per occupation)
    jd_df_unique = (
        df_split[["occupationUri", "jd_text", "preferredLabel", "iscoGroup"]]
        .dropna(subset=["occupationUri", "jd_text"])
        .drop_duplicates(subset=["occupationUri"])
        .reset_index(drop=True)
    )

    jd_corpus_texts = jd_df_unique["jd_text"].tolist()
    jd_corpus_ids   = jd_df_unique["occupationUri"].tolist()

    # Encode all job descriptions 
    jd_emb = model.encode(jd_corpus_texts, convert_to_tensor=True,
                          batch_size=batch_size, show_progress_bar=True)

    # Encode all CV queries
    cv_texts = df_split["cv_text"].tolist()
    cv_emb = model.encode(cv_texts, convert_to_tensor=True,
                          batch_size=batch_size, show_progress_bar=True)

    # Retrieve top-K occupations for each query using cosine similarity
    rows = []
    num_q = cv_emb.size(0)

    for start in range(0, num_q, batch_size):
        end = min(start + batch_size, num_q)
        q_emb = cv_emb[start:end]

        scores = util.cos_sim(q_emb, jd_emb)
        top_scores, top_idx = torch.topk(scores, k=k, dim=1, largest=True, sorted=True)

        # Move tensors to CPU before converting to NumPy / pandas
        top_scores = top_scores.cpu().numpy()
        top_idx = top_idx.cpu().numpy()

        # Store one row per (query, candidate) pair in the top-K list
        for i in range(end - start):
            q_row = df_split.iloc[start + i]
            for rank in range(k):
                cand_occ = jd_corpus_ids[int(top_idx[i, rank])]
                rows.append({
                    "person_id": q_row["person_id"],
                    "t": q_row["t"],
                    "true_occ_uri": q_row["occupationUri"],
                    "candidate_occ_uri": cand_occ,
                    "rank": rank + 1,
                    "sbert_score": float(top_scores[i, rank]),
                })

    # Returns the top-K candidate table and the deduplicated job corpus
    return pd.DataFrame(rows), jd_df_unique

<a class="anchor" id="chapter6"></a>

# 6. Add ESCO overlap features to candidates

</a>

In [ ]:
# Compute ESCO-based skill overlap features for a given CV–occupation pair
def esco_feats(S_cv, occ_uri, occ_to_skills, skill_idf, occ_avg_skill_idf, skill_to_parents):
    # Retrieve skill set for candidate occupation
    S_occ = occ_to_skills.get(occ_uri, set())
    
    # Basic set operations
    inter = S_cv & S_occ
    union = S_cv | S_occ
    overlap = len(inter)

    # IDF-weighted overlap (exact matching)
    idf_sum = float(sum(skill_idf.get(s, 0.0) for s in inter))

    # Hierarchy-expanded overlap: include parent skills of CV skills
    idf_sum_parent = 0.0
    if skill_to_parents is not None and S_cv:
        S_cv_exp = set(S_cv)
        for s in S_cv:
            for p in skill_to_parents.get(s, []):
                S_cv_exp.add(p)

        inter_parent = S_cv_exp & S_occ
        idf_sum_parent = float(sum(skill_idf.get(s, 0.0) for s in inter_parent))

    occ_size = len(S_occ)
    cv_size = len(S_cv)

    # Return feature dictionary
    return {
        "overlap_count": overlap,
        "occ_coverage": overlap / occ_size if occ_size else 0.0,
        "cv_coverage": overlap / cv_size if cv_size else 0.0,
        "jaccard": overlap / len(union) if union else 0.0,
        "idf_overlap_sum": idf_sum,
        "idf_overlap_mean": idf_sum / overlap if overlap else 0.0,
        "occ_avg_skill_idf": float(occ_avg_skill_idf.get(occ_uri, 0.0)),
        "idf_overlap_sum_parent": idf_sum_parent,
    }

In [ ]:
# Add ESCO feature columns to top-K candidate table
def add_esco_features(topk_df, df_split, label_to_uri, occ_to_skills, skill_idf, occ_avg_skill_idf, skill_to_parents):

    # Precompute CV skill sets for efficient lookup
    cv_lookup = build_cv_skill_lookup(df_split, label_to_uri)
    feats_rows = []

    # Compute features for each (CV, candidate occupation) pair
    for _, r in tqdm(topk_df.iterrows(), total=len(topk_df), desc="Adding ESCO features"):
        S_cv = cv_lookup.get((r["person_id"], r["t"]), set())
        feats_rows.append(
            esco_feats(S_cv, r["candidate_occ_uri"], occ_to_skills, skill_idf, occ_avg_skill_idf, skill_to_parents)
        )

    # Concatenate original data with computed feature columns
    feats_df = pd.DataFrame(feats_rows)
    return pd.concat([topk_df.reset_index(drop=True), feats_df], axis=1)

In [ ]:
# Check
row0 = val.iloc[0]
key = (row0["person_id"], row0["t"])

# Build CV skill lookup and extract CV skill set
cv_lookup = build_cv_skill_lookup(val, label_to_uri)
S_cv = cv_lookup[key]

In [ ]:
# Compute ESCO features for the true target occupation
target_occ = row0["occupationUri"]
feats = esco_feats(
    S_cv=S_cv,
    occ_uri=target_occ,
    occ_to_skills=occ_to_skills,
    skill_idf=skill_idf,
    occ_avg_skill_idf=occ_avg_skill_idf,
    skill_to_parents=skill_to_parents,
)
feats

{'overlap_count': 1,
 'occ_coverage': 0.05,
 'cv_coverage': 0.07142857142857142,
 'jaccard': 0.030303030303030304,
 'idf_overlap_sum': 6.311562593298057,
 'idf_overlap_mean': 6.311562593298057,
 'occ_avg_skill_idf': 6.719919675660887,
 'idf_overlap_sum_parent': 6.311562593298057}

In [ ]:
# Compare exact vs hierarchy-expanded overlap
S_occ = occ_to_skills.get(target_occ, set())
exact_inter = len(S_cv & S_occ)

S_cv_exp = set(S_cv)
if skill_to_parents is not None and S_cv:
    for s in S_cv:
        for p in skill_to_parents.get(s, []):
            S_cv_exp.add(p)

parent_inter = len(S_cv_exp & S_occ)

print("Exact overlap count:", exact_inter)
print("Expanded overlap count:", parent_inter)
print("idf_overlap_sum:", feats["idf_overlap_sum"])
print("idf_overlap_sum_parent:", feats["idf_overlap_sum_parent"])

Exact overlap count: 1
Expanded overlap count: 1
idf_overlap_sum: 6.311562593298057
idf_overlap_sum_parent: 6.311562593298057


<a class="anchor" id="chapter7"></a>

# 7. Evaluation metrics

</a>

Evaluation focuses on Recall@K, MRR@K, and nDCG@K. Additional metrics such as Accuracy@K, Precision@K, and MAP are computed for completeness but are not emphasized.

In [ ]:
# Compute ranking metrics from reranked candidate lists
def eval_metrics(df, rank_col="new_rank", k_list=(1, 3, 5, 10, 20)):
    df = df.copy()
    df["is_true"] = (df["candidate_occ_uri"] == df["true_occ_uri"]).astype(int)

    # Retrieve rank of the true occupation for each (person_id, t) query
    true_ranks = df[df["is_true"] == 1].groupby(["person_id", "t"])[rank_col].min()

    # Total number of queries
    num_queries = df.groupby(["person_id", "t"]).ngroups
    out = {}

    # Compute Recall@K (Accuracy@K and Precision@K also computed)
    for K in k_list:
        hit = (true_ranks <= K).sum()
        recall_k = float(hit / num_queries)

        # Under a single-relevance setting, Accuracy@K is equivalent to Recall@K
        out[f"accuracy@{K}"] = recall_k
        out[f"recall@{K}"] = recall_k

        # Precision@K = 1/K if the true item is retrieved within top-K, else 0
        out[f"precision@{K}"] = float(((true_ranks <= K) * (1.0 / K)).sum() / num_queries)

    # Use K=10 as the default cutoff for MRR (or the largest K if 10 is unavailable)
    K_mrr = 10 if 10 in k_list else max(k_list)

    rr = true_ranks.copy()
    rr = rr[rr <= K_mrr]
    mrr = float((1.0 / rr).sum() / num_queries)
    out[f"mrr@{K_mrr}"] = mrr

    # Compute nDCG@K for a single relevant item per query
    def ndcg_single(rank, K):
        return (1.0 / math.log2(rank + 1)) if rank <= K else 0.0

    for K in k_list:
        nd = sum(ndcg_single(r, K) for r in true_ranks.dropna())
        out[f"ndcg@{K}"] = float(nd / num_queries)

    # MAP is included for completeness; under single relevance it is equivalent to MRR
    out[f"map@{K_mrr}"] = mrr

    return out

<a class="anchor" id="chapter8"></a>

# 8. Hybrid-score + tuning (α grid + feature variants)

</a>

In [ ]:
QUERY_COLS = ["person_id", "t"]

In [ ]:
# Compute hybrid score by combining SBERT similarity with normalized ESCO feature
def add_hybrid_score(
    df: pd.DataFrame,
    alpha: float = 0.8,
    esco_col: str = "idf_overlap_sum",
    sbert_col: str = "sbert_score",
    query_cols=QUERY_COLS,
    eps: float = 1e-9,
):
    df = df.copy()

    # Min-max normalize ESCO feature within each query (person_id, t)
    grp = df.groupby(query_cols)[esco_col]
    mn = grp.transform("min")
    mx = grp.transform("max")
    esco_norm = (df[esco_col] - mn) / (mx - mn + eps)

    # Combine SBERT score and normalized ESCO feature
    df[f"{esco_col}_norm"] = esco_norm
    df["hybrid_score"] = alpha * df[sbert_col] + (1 - alpha) * esco_norm

    return df

In [ ]:
# Grid search over ESCO features and alpha values for hybrid reranking (validation set)
def tune_hybrid_on_val(
    val_topk_feat: pd.DataFrame,
    esco_cols,
    alphas,
    k_list=(1, 3, 5, 10, 20),
    query_cols=QUERY_COLS,
):
    all_results = []

    for esco_col in tqdm(esco_cols, desc="ESCO feature variants"):
        for alpha in tqdm(alphas, desc=f"α grid ({esco_col})", leave=False):
            # Compute hybrid scores
            scored = add_hybrid_score(
                val_topk_feat,
                alpha=float(alpha),
                esco_col=esco_col,
                query_cols=query_cols,
            )

            # Rerank candidates within each query
            scored["new_rank"] = scored.groupby(query_cols)["hybrid_score"].rank(ascending=False, method="first")

            # Evaluate ranking performance
            metrics = eval_metrics(scored, rank_col="new_rank", k_list=k_list)
            all_results.append({"esco_col": esco_col, "alpha": float(alpha), **metrics})

    # Return results sorted by Recall@10 (primary metric)
    return pd.DataFrame(all_results).sort_values("recall@10", ascending=False).reset_index(drop=True)

In [ ]:
# Apply best hybrid configuration to a dataset and compute metrics
def apply_best_hybrid(
    df_feat: pd.DataFrame,
    best_row: pd.Series,
    k_list=(1, 3, 5, 10, 20),
    query_cols=QUERY_COLS,
):
    esco_col = best_row["esco_col"]
    alpha = float(best_row["alpha"])

    # Compute hybrid score and rerank
    scored = add_hybrid_score(df_feat, alpha=alpha, esco_col=esco_col, query_cols=query_cols)
    scored["new_rank"] = scored.groupby(query_cols)["hybrid_score"].rank(
        ascending=False, method="first"
    )
    # Evaluate performance
    metrics = eval_metrics(scored, rank_col="new_rank", k_list=k_list)
    
    return scored, metrics

In [ ]:
# Apply fixed hybrid baseline (no tuning)
def run_fixed_hybrid_baseline(
    df_feat: pd.DataFrame,
    alpha=0.8,
    esco_col="idf_overlap_sum",
    k_list=(1, 3, 5, 10, 20),
    query_cols=QUERY_COLS,
):
    # Compute hybrid score and rerank
    scored = add_hybrid_score(df_feat, alpha=float(alpha), esco_col=esco_col, query_cols=query_cols)
    scored["new_rank"] = scored.groupby(query_cols)["hybrid_score"].rank(ascending=False, method="first")
    
    # Evaluate performance
    metrics = eval_metrics(scored, rank_col="new_rank", k_list=k_list)
    
    return scored, metrics

<a class="anchor" id="chapter9"></a>

# 9. Run & Save

</a>

<a class="anchor" id="sub-section-9_1"></a>

## 9.1. Off the shelf SBERT

</a>

In [ ]:
model_off_shelf = SentenceTransformer(MODEL_OFF_SHELF)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Build top-k df
val_topk_df_off, _ = build_topk_df(val, model_off_shelf, k=50, batch_size=256)
test_topk_df_off, _ = build_topk_df(test, model_off_shelf, k=50, batch_size=256)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/346 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/347 [00:00<?, ?it/s]

In [ ]:
# Add ESCO features
val_topk_feat_off = add_esco_features(val_topk_df_off, val, label_to_uri, occ_to_skills, skill_idf, occ_avg_skill_idf, skill_to_parents)
test_topk_feat_off = add_esco_features(test_topk_df_off, test, label_to_uri, occ_to_skills, skill_idf, occ_avg_skill_idf, skill_to_parents)

Adding ESCO features:   0%|          | 0/4424450 [00:00<?, ?it/s]

Adding ESCO features:   0%|          | 0/4432150 [00:00<?, ?it/s]

In [ ]:
val_fixed_off, val_fixed_metrics_off = run_fixed_hybrid_baseline(val_topk_feat_off, alpha=0.8,
                                                                 esco_col="idf_overlap_sum", k_list=(1,3,5,10,20))
test_fixed_off, test_fixed_metrics_off = run_fixed_hybrid_baseline(test_topk_feat_off, alpha=0.8,
                                                                   esco_col="idf_overlap_sum", k_list=(1,3,5,10,20))

In [ ]:
#save_json(val_fixed_metrics_off, f"ESCO_fixedhybrid_offshelf_val_{TIME}.json", out_dir=OUT_DIR)
#save_json(test_fixed_metrics_off, f"ESCO_fixedhybrid_offshelf_test_{TIME}.json", out_dir=OUT_DIR)

<a class="anchor" id="sub-section-9_2"></a>

## 9.2. Tuned SBERT

</a>

In [ ]:
model_tuned = SentenceTransformer(MODEL_TUNED)

In [ ]:
# Build top-k df
val_topk_df_tuned, _ = build_topk_df(val, model_tuned, k=50, batch_size=256)
test_topk_df_tuned, _ = build_topk_df(test, model_tuned, k=50, batch_size=256)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/346 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/347 [00:00<?, ?it/s]

In [ ]:
# Add ESCO features
val_topk_feat_tuned = add_esco_features(val_topk_df_tuned, val, label_to_uri, occ_to_skills, skill_idf, occ_avg_skill_idf, skill_to_parents)
test_topk_feat_tuned = add_esco_features(test_topk_df_tuned, test, label_to_uri, occ_to_skills, skill_idf, occ_avg_skill_idf, skill_to_parents)

Adding ESCO features:   0%|          | 0/4424450 [00:00<?, ?it/s]

Adding ESCO features:   0%|          | 0/4432150 [00:00<?, ?it/s]

In [ ]:
val_fixed_tuned, val_fixed_metrics_tuned = run_fixed_hybrid_baseline(val_topk_feat_tuned, alpha=0.8,
                                                                 esco_col="idf_overlap_sum", k_list=(1,3,5,10,20))
test_fixed_tuned, test_fixed_metrics_tuned = run_fixed_hybrid_baseline(test_topk_feat_tuned, alpha=0.8,
                                                                   esco_col="idf_overlap_sum", k_list=(1,3,5,10,20))

In [ ]:
#save_json(val_fixed_metrics_tuned, f"ESCO_fixedhybrid_tuned_val_{TIME}.json", out_dir=OUT_DIR)
#save_json(test_fixed_metrics_tuned, f"ESCO_fixedhybrid_tuned_test_{TIME}.json", out_dir=OUT_DIR)

<a class="anchor" id="sub-section-9_3"></a>

## 9.3. Hybrid tuning SBERT off shelf

</a>

Since the off-the-shelf SBERT model performed on par with than the fine-tuned variant, and since ESCO augmentation is a post-hoc re-ranking stage independent of model training, hyperparameter tuning of the hybrid score was conducted only for the off-the-shelf SBERT. The tuned hybrid configuration was then evaluated on the test set using the same frozen model.

In [ ]:
ESCO_FEATURES = ["idf_overlap_sum", "idf_overlap_mean", "occ_coverage", "cv_coverage", "jaccard",
                 "overlap_count", "occ_avg_skill_idf", "idf_overlap_sum_parent"]

In [ ]:
tuning_table = tune_hybrid_on_val(val_topk_feat_off, esco_cols=ESCO_FEATURES, alphas=np.linspace(0, 1, 21), k_list=(1,3,5,10,20))

ESCO feature variants:   0%|          | 0/8 [00:00<?, ?it/s]

α grid (idf_overlap_sum):   0%|          | 0/21 [00:00<?, ?it/s]

α grid (idf_overlap_mean):   0%|          | 0/21 [00:00<?, ?it/s]

α grid (occ_coverage):   0%|          | 0/21 [00:00<?, ?it/s]

α grid (cv_coverage):   0%|          | 0/21 [00:00<?, ?it/s]

α grid (jaccard):   0%|          | 0/21 [00:00<?, ?it/s]

α grid (overlap_count):   0%|          | 0/21 [00:00<?, ?it/s]

α grid (occ_avg_skill_idf):   0%|          | 0/21 [00:00<?, ?it/s]

α grid (idf_overlap_sum_parent):   0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
best = tuning_table.iloc[0]
print("Best config (off-the-shelf):", dict(best[["esco_col", "alpha", "recall@10"]]))

Best config (off-the-shelf): {'esco_col': 'occ_coverage', 'alpha': np.float64(0.7000000000000001), 'recall@10': np.float64(0.1399835007741075)}


In [ ]:
val_best_scored_off, val_best_metrics_off = apply_best_hybrid(val_topk_feat_off, best, k_list=(1,3,5,10,20))
test_best_scored_off, test_best_metrics_off = apply_best_hybrid(test_topk_feat_off, best, k_list=(1,3,5,10,20))

In [ ]:
#tuning_table.to_csv(f"{OUT_DIR}/ESCO_tuning_offshelf_VAL_{TIME}.csv", index=False)
#save_json(val_best_metrics_off, f"ESCO_tunedhybrid_offshelf_val_{TIME}.json", out_dir=OUT_DIR)
#save_json(test_best_metrics_off, f"ESCO_tunedhybrid_offshelf_test_{TIME}.json", out_dir=OUT_DIR)

<a class="anchor" id="sub-section-9_4"></a>

## 9.4. Save val_topk_feat_off and test_topk_feat_off

</a>

In [ ]:
KEEP_COLS_TOPK_FEAT = [
    "person_id", "t",
    "true_occ_uri", "candidate_occ_uri",
    "rank", "sbert_score",
    "overlap_count", "occ_coverage", "cv_coverage", "jaccard",
    "idf_overlap_sum", "idf_overlap_mean", "occ_avg_skill_idf",
    "idf_overlap_sum_parent",
]

INT_COLS = ["person_id", "t", "rank", "overlap_count"]
FLOAT_COLS = [
    "sbert_score", "occ_coverage", "cv_coverage", "jaccard",
    "idf_overlap_sum", "idf_overlap_mean", "occ_avg_skill_idf",
    "idf_overlap_sum_parent",
]
CAT_COLS = ["true_occ_uri", "candidate_occ_uri"]

In [ ]:
# Val
val_topk_feat_off_slim = val_topk_feat_off.loc[:, KEEP_COLS_TOPK_FEAT].copy()

for c in INT_COLS:
    if c in val_topk_feat_off_slim.columns:
        val_topk_feat_off_slim[c] = pd.to_numeric(val_topk_feat_off_slim[c], downcast="integer")

for c in FLOAT_COLS:
    if c in val_topk_feat_off_slim.columns:
        val_topk_feat_off_slim[c] = pd.to_numeric(val_topk_feat_off_slim[c], downcast="float")

for c in CAT_COLS:
    if c in val_topk_feat_off_slim.columns:
        val_topk_feat_off_slim[c] = val_topk_feat_off_slim[c].astype("category")

In [ ]:
# Save validation top-K features
val_path = DATA_DIR / "TopK" / "val_topk_feat_off.parquet"
val_path.parent.mkdir(parents=True, exist_ok=True)
val_topk_feat_off_slim.to_parquet(val_path, index=False)

In [ ]:
# Test
test_topk_feat_off_slim = test_topk_feat_off.loc[:, KEEP_COLS_TOPK_FEAT].copy()

for c in INT_COLS:
    if c in test_topk_feat_off_slim.columns:
        test_topk_feat_off_slim[c] = pd.to_numeric(test_topk_feat_off_slim[c], downcast="integer")

for c in FLOAT_COLS:
    if c in test_topk_feat_off_slim.columns:
        test_topk_feat_off_slim[c] = pd.to_numeric(test_topk_feat_off_slim[c], downcast="float")

for c in CAT_COLS:
    if c in test_topk_feat_off_slim.columns:
        test_topk_feat_off_slim[c] = test_topk_feat_off_slim[c].astype("category")

In [ ]:
# Save test top-K features
test_path = DATA_DIR / "TopK" / "test_topk_feat_off.parquet"
test_path.parent.mkdir(parents=True, exist_ok=True)
test_topk_feat_off_slim.to_parquet(test_path, index=False)